In [1]:
import os
import random
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import optuna
import time

PROJECT_ROOT = Path.cwd().parent
SRC_PATH = PROJECT_ROOT / "src"
DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"

if str(SRC_PATH) not in sys.path:
    sys.path.append(str(SRC_PATH))

from utils import eda_toolkit as toolkit
from utils import build_features as bf
from utils import trainer_toolkit as tt
from utils import test_toolkit as ts

from eval.scorer import DemandScorer
from eval.holdout import build_holdout_dataset, evaluate_on_holdout
from models.baselines import seasonal_naive, historical_median
from models.lgbm_tweedie_wrapper import LightGBMTweedieWrapper
from trainer.walk_forward import WalkForwardTrainer

warnings.filterwarnings("ignore")

SEED = 42
os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)

RESULTS_PATH = PROJECT_ROOT / "results" / "lgbm"
MODELS_PATH = PROJECT_ROOT / "models"

RESULTS_PATH.mkdir(parents=True, exist_ok=True)
MODELS_PATH.mkdir(parents=True, exist_ok=True)

train = pd.read_parquet(DATA_PROCESSED / "train_k54.parquet")
holdout = pd.read_parquet(DATA_PROCESSED / "holdout_k54.parquet")

feature_builder = lambda df: bf.build_features(df, dropna=True)

assert train.index.max() < holdout.index.min(), "Solapamiento temporal train/holdout"

print("=" * 70)
print(f"Train:   {len(train):>6} filas  ({train.index[0]} -> {train.index[-1]})")
print(f"Holdout: {len(holdout):>6} filas  ({holdout.index[0]} -> {holdout.index[-1]})")
print("=" * 70)

Train:    56992 filas  (2023-01-01 04:30:00 -> 2025-05-26 17:15:00)
Holdout:  14248 filas  (2025-05-26 17:30:00 -> 2025-12-31 20:30:00)


# Introducción

El Gradient Boosting con LightGBM es un marco de aprendizaje supervisado que construye un ensamble aditivo de árboles de decisión mediante la optimización secuencial de una función de pérdida diferenciable. A diferencia de los modelos que imponen una ecuación paramétrica para la evolución de la media condicional, LightGBM aprende la relación entre las variables predictoras y la respuesta directamente a partir de los datos, sin asumir una forma funcional específica.

En el contexto de la demanda de pasajeros por franja horaria, esta arquitectura ofrece una alternativa a los modelos recurrentes y a los enfoques econométricos tradicionales, al permitir que la relación entre el calendario, la estacionalidad, los rezagos y la demanda futura sea aprendida directamente de los datos. La naturaleza de la respuesta, sin embargo, impone una restricción sobre la función de pérdida: la serie presenta sobredispersión marcada en las horas de mayor operación y una proporción apreciable de franjas con cero pasajeros, estructura que una pérdida cuadrática o una verosimilitud Poisson subestimarían. Por esta razón se adopta la distribución Tweedie como familia de verosimilitud del objetivo, cuya forma compuesta Poisson-Gamma asigna masa puntual en cero y una densidad continua para los valores positivos.

El objetivo de esta sección es presentar, entrenar y evaluar modelos LightGBM con pérdida Tweedie para la predicción de la demanda de pasajeros por franja horaria en la ruta K54. Dado que la operación requiere pronósticos a múltiples escalas temporales, el problema se formula como una tarea de predicción directa multi-horizonte. Para cada intervalo operativo solicitado (30, 60 y 120 minutos) se entrena y optimiza un modelo LightGBM independiente, cada uno con su propio objetivo agregado y su propia validación. La evaluación sobre el *holdout* replica las condiciones de despliegue operativo mediante un reentrenamiento rodante por bloques con ventana deslizante. Este esquema respeta la partición cronológica y las restricciones metodológicas del proyecto: emplea exclusivamente variables disponibles en el instante de decisión y previene cualquier fuga de información entre conjuntos.

### Arquitectura

LightGBM construye un modelo aditivo de árboles de decisión que aproxima la función objetivo de forma secuencial. Para un conjunto de $N$ observaciones $\{\mathbf{x}_i, y_i\}_{i=1}^{N}$, el modelo final se expresa como la suma de $M$ árboles $h_m(\cdot)$ escalados por una tasa de aprendizaje $\nu$:

$$
F_M(\mathbf{x}_i) = \sum_{m=1}^{M} \nu \, h_m(\mathbf{x}_i),
$$

donde cada árbol $h_m \in \mathcal{H}$ es una función escalonada sobre el espacio de predictores. El ajuste es *forward stagewise*: en la iteración $m$, el modelo previo $F_{m-1}$ se mantiene fijo y se añade un nuevo árbol que corrige sus errores,

$$
F_m(\mathbf{x}) = F_{m-1}(\mathbf{x}) + \nu \, h_m(\mathbf{x}).
$$

El nuevo árbol se obtiene minimizando una expansión de segundo orden de la pérdida $\mathcal{L}$ alrededor de la predicción actual. Definiendo el gradiente $g_{im}$ y el hessiano $h_{im}$ de la pérdida respecto a $F_{m-1}(\mathbf{x}_i)$,

$$
g_{im} = \left. \frac{\partial \mathcal{L}(y_i, F)}{\partial F} \right|_{F = F_{m-1}(\mathbf{x}_i)}, \qquad
h_{im} = \left. \frac{\partial^2 \mathcal{L}(y_i, F)}{\partial F^2} \right|_{F = F_{m-1}(\mathbf{x}_i)},
$$

las hojas del árbol se calculan con la solución de Newton,

$$
w_{j}^{*} = - \frac{\sum_{i \in I_j} g_{im}}{\sum_{i \in I_j} h_{im} + \lambda},
$$

donde $I_j$ es el conjunto de observaciones que caen en la hoja $j$ y $\lambda$ es el término de regularización $\ell_2$. La ganancia de una partición que divide $I$ en $I_L$ e $I_R$ se evalúa como:

$$
\mathcal{G} = \frac{1}{2}\left[
\frac{\left(\sum_{i \in I_L} g_{im}\right)^2}{\sum_{i \in I_L} h_{im} + \lambda} +
\frac{\left(\sum_{i \in I_R} g_{im}\right)^2}{\sum_{i \in I_R} h_{im} + \lambda} -
\frac{\left(\sum_{i \in I} g_{im}\right)^2}{\sum_{i \in I} h_{im} + \lambda}
\right].
$$

Bajo el objetivo `tweedie`, LightGBM asume que la respuesta sigue una distribución Tweedie con parámetro de potencia $p$ y utiliza una función de enlace logarítmica, de modo que la media condicional se obtiene como $\hat{\mu}(\mathbf{x}) = \exp(F(\mathbf{x}))$. La pérdida que se minimiza es la desviación negativa de la log-verosimilitud Tweedie,

$$
\mathcal{L}_{\text{Tweedie}}(y, \hat{\mu}) = - \left[ y \, \frac{\hat{\mu}^{1-p}}{1-p} - \frac{\hat{\mu}^{2-p}}{2-p} \right].
$$

:::{admonition} Información: Distribución Tweedie
:class: info

La distribución Tweedie con parámetro de potencia $p$ pertenece a la familia exponencial y se caracteriza por la relación entre la media $\mu$ y la varianza:

$$
\operatorname{Var}(y) = \phi \, \mu^{p}, \qquad 1 < p < 2.
$$

Valores de $p$ cercanos a 1 aproximan la verosimilitud a una Poisson; valores cercanos a 2 la aproximan a una Gamma; los valores intermedios corresponden a la familia compuesta Poisson-Gamma, que admite masa en cero y una componente continua positiva. En este proyecto, el parámetro $p$ se estima empíricamente a partir de la relación varianza-media de la serie. El valor resultante se fija como constante durante el entrenamiento para garantizar la estabilidad y comparabilidad de la métrica.
:::

Con la función de enlace logarítmica, las derivadas de la pérdida respecto a la predicción del modelo son:

$$
\frac{\partial \mathcal{L}}{\partial F} = -y \, e^{(1-p)F} + e^{(2-p)F}, \qquad
\frac{\partial^2 \mathcal{L}}{\partial F^2} = -y(1-p) \, e^{(1-p)F} + (2-p) \, e^{(2-p)F}.
$$

Estas expresiones alimentan el gradiente $g_{im}$ y el hessiano $h_{im}$ en cada iteración del boosting y son las que permiten que el modelo pondere de forma diferenciada las franjas con cero pasajeros y las franjas de alta demanda. Para $1 < p < 2$, el hessiano es estrictamente positivo, lo que garantiza la convexidad local del subproblema de optimización en cada hoja.

El modelo completo $F_M(\cdot)$ asigna a cada vector de predictores $\mathbf{x}$ (hora, día de la semana, mes, codificación cíclica, estacionalidad, rezagos y tendencia temporal) una predicción de la media condicional de pasajeros acumulados en la ventana objetivo, $\hat{\mu}(\mathbf{x}) = \exp(F_M(\mathbf{x}))$.

### Protocolo de entrenamiento y validación

Se entrena un modelo LightGBM independiente por cada horizonte de predicción (30, 60 y 120 minutos, equivalentes a $h \in \{2, 4, 8\}$ franjas). El *target* de cada modelo es la suma rodante de pasajeros desde el instante de decisión hasta el final del intervalo, compartiendo todos el mismo conjunto de 23 variables predictoras. Dado que la agregación temporal altera la dispersión de la demanda, el parámetro de potencia $p$ de Tweedie se estima empíricamente de forma independiente para cada horizonte y se fija como constante durante su entrenamiento. Cada intervalo se predice en un único paso, sin usar las predicciones de un horizonte como insumo de otro.

Para mitigar el cambio de régimen estructural entre 2023 y 2025 y evitar sesgos por sobrepredicción, se implementó una ventana deslizante (*sliding window*). El hiperparámetro `window` define la historia reciente que observa el modelo; se exploraron tamaños entre 18,000 y 26,000 franjas (de 8 a 12 meses), con un calentamiento inicial fijado en 26,000.

La validación en `train.parquet` usa un esquema *walk-forward* de 5 pliegues, aplicando un *embargo* de $h-1$ franjas en la frontera para evitar fugas de información. Para la evaluación final, el `holdout.parquet` replica esta mecánica avanzando cronológicamente y reentrenando el modelo por bloques con las últimas `window` franjas disponibles.

La implementación utiliza `LightGBM` (v4.6) con el objetivo `tweedie`. El *early stopping* interno se evalúa sobre una partición cronológica 70/30 dentro de cada bloque de entrenamiento, respetando también el *embargo* del horizonte.

## Ventana de predicción de 30 minutos

### Entrenamiento

Para determinar el parámetro de potencia $p$ en la muestra de entrenamiento, se aplicó un estimador por momentos basado en la relación teórica entre la varianza y la media de la distribución Tweedie:

$$
\log(\operatorname{Var}(y)) = \log(\phi) + p \log(\mu)
$$

Al agrupar las observaciones por hora y día de la semana, y calcular la pendiente de esta regresión lineal sobre las medias y varianzas resultantes, se obtuvo un valor de $p=1.4$. Al contrastar este ajuste con los datos reales, el análisis visual revela que la curva teórica no alcanza la altura máxima del pico empírico y presenta desviaciones en los extremos del Q-Q plot. Una hipótesis plausible para explicar este comportamiento es la alta heterogeneidad de la serie: al intentar abarcar la gran dispersión generada por la mezcla de regímenes operativos (horas pico, valles y fines de semana), la distribución teórica se aplana en el centro y engrosa sus colas. Dicho sesgo marginal no afecta el desempeño predictivo del modelo LightGBM, ya que la función de pérdida Tweedie se aplica de forma condicional sobre las hojas del árbol, capturando la dinámica local de cada franja en lugar del promedio global.

In [9]:
warmup_data = train["pasajeros"].iloc[:26000]
pasajeros_h2_warmup = warmup_data.rolling(window=2).sum().shift(-1).dropna()

_, _ = tt.plot_tweedie_fit(
    series=pasajeros_h2_warmup, 
    p=p_warmup,
    titles=("Distribución de Pasajeros (Warmup)", "Q-Q Plot (Warmup)"),
    xlab="Pasajeros acumulados en 30 min (h=2)",
    pads=(15, 15, 12), 
    legend_bbox=(1.0, 0.95)
)

Para la selección de hiperparámetros del modelo LightGBM correspondiente al horizonte de 30 minutos ($h = 2$ franjas), se implementó una validación cruzada temporal de 5 pliegues sobre el conjunto de entrenamiento. La búsqueda de Optuna exploró la complejidad de los árboles de decisión, los parámetros de regularización y el tamaño de la ventana de historia reciente.

El modelo recibe directamente una matriz tabular con las 23 variables predictoras (calendario, codificación cíclica, estacionalidad, tendencia temporal y rezagos precalculados). Para evitar el sesgo de anticipación (*look-ahead bias*) y adaptarse al cambio de régimen estructural de la demanda, se implementó una estrategia de ventana deslizante (*sliding window*). En cada pliegue, el modelo se reentrena con un bloque de historia delimitado por el hiperparámetro `window` y se valida con el bloque cronológico inmediatamente posterior, aplicando un *embargo* de una franja ($h - 1 = 1$) para garantizar la integridad temporal de los datos.

El espacio de búsqueda incluyó el tamaño de la ventana de entrenamiento (`window`), el número máximo de hojas por árbol (`num_leaves`), la tasa de aprendizaje (`learning_rate`), el mínimo de datos y de suma de hessianos por hoja (`min_data_in_leaf`, `min_sum_hessian_in_leaf`), las fracciones y frecuencia de submuestreo (`bagging_fraction`, `bagging_freq`, `feature_fraction`) y las penalizaciones por regularización $\ell_1$ y $\ell_2$. El entrenamiento se configuró con un máximo de 5,000 árboles y una paciencia de 300 iteraciones para la parada temprana (*early stopping*) interna, la cual se evalúa sobre el último 10% cronológico de cada bloque de entrenamiento, respetando también el embargo de una franja. El desempeño se evaluó mediante la desviación Tweedie, fijando el parámetro de potencia en $p = 1.4311$ (estimado exclusivamente con el bloque de calentamiento para evitar fuga de información) y aplicando un límite superior de 100 por observación para mitigar el efecto de eventos atípicos impredecibles.

In [6]:
WINDOW_MAX = 26000

def lgbm_search_space(trial: optuna.Trial) -> dict:
    return {
        "window": trial.suggest_categorical(
            "window", [18000, 20000, 22000, 24000, WINDOW_MAX]
        ),
        "num_leaves": trial.suggest_int("num_leaves", 15, 300, log=True),
        "learning_rate": trial.suggest_float("learning_rate", 1e-3, 0.2, log=True),
        "min_data_in_leaf": trial.suggest_int("min_data_in_leaf", 10, 500, log=True),
        "feature_fraction": trial.suggest_float("feature_fraction", 0.4, 1.0, step=0.05),
        "bagging_fraction": trial.suggest_float("bagging_fraction", 0.4, 1.0, step=0.05),
        "bagging_freq": trial.suggest_int("bagging_freq", 1, 10),
        "lambda_l1": trial.suggest_float("lambda_l1", 1e-8, 10.0, log=True),
        "lambda_l2": trial.suggest_float("lambda_l2", 1e-8, 50.0, log=True),
        "min_sum_hessian_in_leaf": trial.suggest_float(
            "min_sum_hessian_in_leaf", 1e-3, 10.0, log=True
        ),
    }

exec_id = 8
exec_dir = RESULTS_PATH / f"exec_{exec_id}"
exec_dir.mkdir(parents=True, exist_ok=True)

trainer = WalkForwardTrainer(
    model_class=LightGBMTweedieWrapper,
    scorer=DemandScorer(),
    output_dir=exec_dir,
    feature_builder=feature_builder,
    horizon_slots=2,
    n_splits=5,
    warmup=WINDOW_MAX,
    embargo=1,
    n_trials=100,
    suggest_params_fn=lgbm_search_space,
    early_stop_rounds=30,
    inner_val_frac=0.1,
    model_kwargs={
        "tweedie_variance_power": 1.4311,
        "n_estimators": 5000,
        "early_stopping_rounds": 300,
        "num_threads": 8,
        "seed": 42,
    },
)

study = trainer.run(train)
best_params = dict(study.best_params)

  0%|          | 0/100 [00:00<?, ?it/s]

Respecto a la eficiencia del algoritmo de búsqueda, la optimización se detuvo en el ensayo 58 sin llegar a la totalidad de los 100 ensayos programados, debido a la activación del mecanismo de parada temprana (30 ensayos consecutivos sin mejora tras el ensayo 27). El mecanismo de poda (*pruning*) descartó 5 configuraciones, confirmando la utilidad del *Median Pruner* para abortar tempranamente arquitecturas con bajo desempeño predictivo durante las primeras particiones. De los 53 ensayos completados, la mejor configuración alcanzó una Desviación Tweedie de 3.2053.

El entrenamiento efectivo de los árboles a través de los 265 pliegues evaluados (53 ensayos completados $\times$ 5 pliegues) acumuló un tiempo neto de 30.43 minutos (1,825.71 segundos), equivalente a un promedio de 6.38 segundos por pliegue bajo el esquema de ventana deslizante. La ejecución completa tuvo una duración de 35 minutos y 46 segundos (2,146 segundos); la diferencia de aproximadamente 5.34 minutos (320.29 segundos) respecto al tiempo acumulado de los pliegues responde al ajuste inicial sobre el bloque de calentamiento en cada ensayo y a la evaluación de las métricas de validación en memoria. Dado que LightGBM realiza la inferencia de forma vectorizada, la generación de pronósticos sobre el conjunto de prueba (*holdout*) se ejecuta en fracciones de segundo en cada bloque rodante.

In [7]:
resumen_data = []

exec_id = 8
exec_dir = RESULTS_PATH / f"exec_{exec_id}"

tiempo_total_rep = 0.0
tiempo_promedio_fold = 0.0
num_folds = 0
total_trials, completos, podados, fallidos = 0, 0, 0, 0
mejor_tweedie = "-"

csv_files = list(exec_dir.glob("*_results.csv"))
if csv_files:
    df_csv = pd.read_csv(csv_files[0])
    tiempo_total_rep = df_csv["train_time_sec"].sum()
    tiempo_promedio_fold = df_csv["train_time_sec"].mean()
    num_folds = len(df_csv)
else:
    print(f"No se encontró el CSV en {exec_dir}")

db_path = exec_dir / "optuna_study.db"
if db_path.exists():
    try:
        storage_url = f"sqlite:///{db_path.resolve()}"
        study = optuna.load_study(
            study_name="LightGBMTweedieWrapper_h2_study", 
            storage=storage_url
        )

        total_trials = len(study.trials)
        completos = len([t for t in study.trials if t.state == optuna.trial.TrialState.COMPLETE])
        podados  = len([t for t in study.trials if t.state == optuna.trial.TrialState.PRUNED])
        fallidos = len([t for t in study.trials if t.state == optuna.trial.TrialState.FAIL])
        
        mejor_tweedie = round(study.best_value, 4)

    except Exception as e:
        print(f"Error leyendo BD en {exec_dir}: {e}")
else:
    print(f"No se encontró la BD en {db_path}")

resumen_data.append({
    "Total Trials": total_trials,
    "Completos": completos,
    "Podados": podados,
    "Fallidos": fallidos,
    "Mejor Tweedie Dev": mejor_tweedie,
    "Tiempo Total (s)": round(tiempo_total_rep, 2),
    "Tiempo por Fold (s)": round(tiempo_promedio_fold, 3)
})

df_search_params = pd.DataFrame(resumen_data)
tt.display_styled_df(df=df_search_params, decimals=4, align_left_first=False)

Total Trials,Completos,Podados,Fallidos,Mejor Tweedie Dev,Tiempo Total (s),Tiempo por Fold (s)
58,53,5,0,3.2053,1825.7100,6.3840


El ranking de configuraciones mostró que las cinco mejores opciones presentaron una Desviación Tweedie en torno a 3.21. Dado que la variabilidad entre los pliegues de validación se mantuvo relativamente alta ($\sigma \approx 1.06$), las diferencias de milésimas entre estas configuraciones representan un empate estadístico. Sin embargo, el proceso mostró una notable estabilidad en la selección de hiperparámetros: los cinco mejores modelos convergieron de manera unánime hacia el límite superior de ventana de historia evaluado de 26000 franjas, un submuestreo pronunciado de predictores (`feature_fraction` entre 0.40 y 0.45), fracciones moderadas de muestreo de filas (`bagging_fraction` entre 0.60 y 0.65), tasas de aprendizaje bajas y penalizaciones $\ell_1$ y $\ell_2$ cercanas a cero.

La selección consistente de 26000 franjas (equivalentes a más de un año calendario, dado que cada franja representa 15 minutos) permite maximizar la cobertura del ciclo estacional anual y disponer de mayor volumen de ejemplos históricos para estimar los patrones semanales y de calendario sin degradar el ajuste sobre los regímenes recientes. La combinación de tasas de aprendizaje bajas con un submuestreo estricto de variables y árboles de complejidad intermedia resultó suficiente para regularizar el ensamble y mitigar el sobreajuste sin requerir penalizaciones explícitas sobre los pesos de las hojas.

In [8]:
exec_dir = RESULTS_PATH / "exec_8"
df_lgbm = pd.read_csv(exec_dir / "LightGBMTweedieWrapper_results.csv")

df_lgbm["lr"] = df_lgbm["learning_rate"].apply(lambda x: f"{x:.4f}")
df_lgbm["l1"] = df_lgbm["lambda_l1"].apply(lambda x: f"{x:.1e}")
df_lgbm["l2"] = df_lgbm["lambda_l2"].apply(lambda x: f"{x:.1e}")

cols_agrupar = [
    "window", 
    "num_leaves", 
    "min_data_in_leaf", 
    "feature_fraction", 
    "bagging_fraction", 
    "lr", 
    "l1",
    "l2"
]

df_agg = df_lgbm.groupby(cols_agrupar)["tweedie_deviance"].agg(["mean", "std"]).reset_index()

df_agg["Tweedie Deviance"] = df_agg.apply(lambda x: f"{x['mean']:.3f} ± {x['std']:.3f}", axis=1)

df_top5 = df_agg.sort_values(by="mean").head(5)

df_top5 = df_top5.rename(columns={
    "window": "Ventana",
    "num_leaves": "Hojas",
    "min_data_in_leaf": "Min Data Hoja",
    "feature_fraction": "Feat. Frac",
    "bagging_fraction": "Bagging Frac",
    "lr": "LR",
    "l1": "L1",
    "l2": "L2"
})

df_final = df_top5[
    ["Ventana", "Hojas", "Min Data Hoja", "Feat. Frac", "Bagging Frac", "LR", "L1", "L2", "Tweedie Deviance"]
]

tt.display_styled_df(df=df_final, decimals=4, align_left_first=False)

Ventana,Hojas,Min Data Hoja,Feat. Frac,Bagging Frac,LR,L1,L2,Tweedie Deviance
"26,000",42,17,0.4000,0.6500,0.0095,2.7e-04,2.2e-07,3.205 ± 1.058
"26,000",85,17,0.4000,0.6000,0.0086,5.3e-03,1.4e-06,3.216 ± 1.071
"26,000",34,14,0.4000,0.6500,0.0070,1.7e-01,4.7e-07,3.218 ± 1.064
"26,000",87,14,0.4000,0.6500,0.0067,6.7e-04,2.3e-06,3.221 ± 1.076
"26,000",30,11,0.4500,0.6500,0.0186,3.4e-03,8.3e-07,3.221 ± 1.048


La selección del modelo ganador entre las cinco mejores configuraciones se resolvió a favor de la primera (Desviación Tweedie de 3.205). Además de obtener el menor promedio de pérdida y una ligera reducción en la variabilidad entre pliegues frente a la segunda opción ($\sigma = 1.058$ frente a $\sigma = 1.071$), el modelo seleccionado ofrece un balance superior entre capacidad expresiva y parsimonia: reduce a menos de la mitad la complejidad estructural de los árboles (42 hojas frente a 85 hojas), mantiene un submuestreo estricto de predictores (`feature_fraction` = 0.40) y emplea una tasa de aprendizaje ligeramente superior (0.0095 frente a 0.0086) que favorece una convergencia eficiente sin sacrificar generalización.

In [11]:
best_params = {
    "window": 26000,
    "num_leaves": 42,
    "learning_rate": 0.009484004689773003,
    "min_data_in_leaf": 17,
    "feature_fraction": 0.4,
    "bagging_fraction": 0.65,
    "bagging_freq": 8,
    "lambda_l1": 0.00026621217695759153,
    "lambda_l2": 2.1604482856110947e-07,
    "min_sum_hessian_in_leaf": 0.0020565134495896358,
}

window = best_params["window"]
model_params = {k: v for k, v in best_params.items() if k != "window"}

final_kwargs = {
    "tweedie_variance_power": 1.4311,
    "n_estimators": 5000,
    "early_stopping_rounds": 300,
    "inner_val_frac": 0.3,
    "num_threads": 8,
    "seed": 42,
}

trainer = WalkForwardTrainer(
    model_class=LightGBMTweedieWrapper,
    scorer=DemandScorer(),
    output_dir=RESULTS_PATH / "exec_8",
    feature_builder=feature_builder,
    horizon_slots=2,
    n_splits=5,
    warmup=26000,
    embargo=1,
    n_trials=100,
    model_kwargs={},
)

X_train_full, y_train_full, _, _ = trainer._build_dataset(train)

model_production = LightGBMTweedieWrapper(**model_params, **final_kwargs)
model_production.fit(X_train_full.iloc[-window:], y_train_full[-window:])

Para evaluar la estructura de los residuos tanto en el conjunto de entrenamiento (*In-Sample*) como en el de prueba (*Out-of-Sample*), a partir de las demandas predichas y las observaciones reales, se calcularon los residuos y su versión estandarizada mediante las siguientes expresiones:

$$
e_t = y_t - \hat{y}_t, \qquad z_t = \frac{e_t - \bar{e}}{s_e}
$$

donde $y_t$ es la demanda observada, $\hat{y}_t$ es la predicción del modelo, $\bar{e}$ es la media muestral de los residuos y $s_e$ es su desviación estándar.

Debido a que los estadísticos de Ljung-Box y ARCH-LM escalan linealmente con el tamaño de la muestra, evaluar la totalidad de la ventana activa de entrenamiento ($26000$ franjas) introduce un efecto de sobre-potencia (*over-power*) que infla los estadísticos ante cualquier dependencia residual. Para atenuar este sesgo sin alterar la secuencia cronológica de los rezagos, el diagnóstico se realizó sobre el último bloque continuo de entrenamiento ($N = 1000$ franjas). A pesar de esta reducción muestral, las pruebas de Ljung-Box (autocorrelación) y ARCH-LM (heterocedasticidad) rechazaron sus hipótesis nulas en todos los rezagos evaluados ($p < .001$).

In [25]:
preds_train_final = model_production.predict_in_sample(X_train_full[-window:])
residuals_train = y_train_full[-window:] - preds_train_final

block_size = 1000
residuals_block = residuals_train[-block_size:]
std_residuals_block = (
    (residuals_block - residuals_block.mean())
    / residuals_block.std(ddof=1)
)

diag_train = tt.run_model_diagnostics(
    std_residuals_block,
    lags=(10, 20, 30),
)

tt.display_styled_df(
    df=diag_train,
    p_value_cols=["p-valor", "p-ajustado"],
    align_left_first=True,
)

Prueba,Estadístico,p-valor,p-ajustado (holm),Conclusión (α=0.05)
Ljung-Box (Lag 10),261.258,< .001,< .001,Autocorrelación Significativa
Ljung-Box (Lag 20),278.194,< .001,< .001,Autocorrelación Significativa
Ljung-Box (Lag 30),307.470,< .001,< .001,Autocorrelación Significativa
ARCH-LM (Lag 10),131.848,< .001,< .001,Efectos ARCH Significativos
ARCH-LM (Lag 20),134.227,< .001,< .001,Efectos ARCH Significativos
ARCH-LM (Lag 30),142.000,< .001,< .001,Efectos ARCH Significativos


La persistencia de autocorrelación residual obedece a la exclusión operativa de los rezagos intradía recientes ($k < 32$) y a la superposición del horizonte de predicción ($h = 2$), factores que trasladan los choques de corto plazo al término de error. La detección de efectos ARCH refleja la relación media-varianza propia de la demanda de pasajeros $\left(\operatorname{Var}(y) = \phi\,\mu^{1.4311}\right)$, donde la dispersión aumenta en las franjas de alta afluencia. Este comportamiento es característico de los conteos de transporte con sobredispersión, y la función de pérdida Tweedie logró asimilarlo sin comprometer la capacidad predictiva de la media condicional.

La inspección visual evidenció que el modelo LightGBM-Tweedie reprodujo los ciclos estacionales regulares de la operación de SOBUSA, las mesetas de demanda en días hábiles y los valles de fin de semana. También capturó las contracciones atípicas registradas entre 2024 y 2025, incluyendo las caídas de cierre de año, Semana Santa y la interrupción operativa de septiembre de 2024.

In [34]:
ts.plot_temporal_fit_panel(
    dates=X_train_full.index[-window:],
    y_true=y_train_full[-window:],
    y_pred=preds_train_final,
    zoom_left=("2025-02-02", "2025-02-08"),
    zoom_right=("2025-02-28", "2025-02-28"),
    text_left="Febrero 2025",
    text_right="28 Febrero 2025",
    title="Demanda de pasajeros 2024-2025",
    xlab_left="Día del mes",
    xlab_right="Hora del día",
    ylab_top="Pasajeros por día",
    ylab_zoom="Pasajeros por franja",
    zoom_ylim_pad=(0.10, 0.12), top_ylim = (-2000, 40000),
    hspace=0.1, wspace = 0.05
)

A partir de febrero de 2025 se presentó una subestimación sistemática de la envolvente diaria: la demanda laboral real repuntó hacia niveles cercanos a $2.8 \times 10^4$ pasajeros diarios, mientras que la predicción se estabilizó alrededor de $2.3\text{--}2.4 \times 10^4$. Este desfase responde a la contracción estructural de la demanda documentada previamente en el análisis exploratorio (EDA) y a la amplitud de la ventana de entrenamiento ($26000$ franjas): al abarcar más de un año calendario, el ensamble pondera con mayor peso los regímenes más moderados de mediados y finales de 2024, lo que, sumado al suavizado hacia la media condicional propio de los árboles de decisión, ancla las predicciones de los primeros meses de 2025 por debajo del nuevo techo operativo.

En el detalle intradía de la semana del 2 al 8 de febrero de 2025 y de la jornada individual seleccionada, esta subestimación se concentró en la cúspide de los picos matutinos (alrededor de $800\text{--}930$ pasajeros por franja frente a máximos reales de $1050\text{--}1200$ entre el 3 y el 7 de febrero, y cerca de $570$ frente a más de $900$ en el detalle diario) y en las mesetas de media mañana y mediodía. A pesar de este suavizamiento en las crestas de alta frecuencia, la estructura basada en árboles capturó con precisión la transición matutina, el repunte vespertino y el descenso nocturno a lo largo de la serie.

### Evaluación

La evaluación del rendimiento del modelo LightGBM-Tweedie contrasta su capacidad de ajuste sobre los datos observados con su capacidad predictiva sobre demanda no vista. Mientras que la evaluación dentro de la muestra (*In-Sample*, sobre la ventana activa $w = 26{,}000$ franjas del cierre de entrenamiento) verifica la convergencia del algoritmo mediante un ajuste estático, la evaluación fuera de la muestra (*Out-of-Sample*, conjunto *holdout* del `2025-05-26` al `2025-12-31`, $N = 14{,}247$ franjas) determina la utilidad operativa real ante nuevos regímenes de movilidad mediante reentrenamiento secuencial.

Para diagnosticar la presencia de sobreajuste (*overfitting*) o subajuste (*underfitting*), se aplicaron los criterios establecidos en el protocolo experimental. Estos criterios se basan en el error relativo porcentual entre las métricas de prueba y entrenamiento, denotado como:

$$
\Delta_{\text{rel}} (\%) = \frac{M_{\text{test}} - M_{\text{train}}}{|M_{\text{train}}|} \times 100
$$

Se estableció un umbral de tolerancia relativo de $|\Delta_{\text{rel}}| = 5\%$ sobre la función de pérdida principal (*Tweedie Deviance*, con parámetro de potencia óptimo $p = 1.4311$). Un incremento relativo superior a este margen evidenciaría degradación en la capacidad de generalización.

En el contexto del pronóstico de demanda de pasajeros bajo verosimilitud Tweedie, debe considerarse la naturaleza asimétrica de esta función de pérdida y su sensibilidad diferenciada entre franjas de baja afluencia (incluyendo cierres operativos en cero) y horas pico. Dado que un desplazamiento puntual en la demanda real afecta de forma no lineal las métricas agregadas, la lectura de $\Delta_{\text{rel}}$ se complementó con métricas de escala (`MAE`, `RMSE`), métricas relativas (`SMAPE`, `MASE`), el diagnóstico de sesgo (`Bias`) y la inspección visual de las trayectorias diarias e intradía.

Las predicciones fuera de muestra se obtuvieron ejecutando el método `forecast` de `LightGBMTweedieWrapper` sobre la serie continua concatenada (garantizando la disponibilidad exacta de los rezagos operativos sin pérdida de observaciones iniciales ni fuga de información). Este procedimiento aplicó un esquema de reentrenamiento periódico por bloques (`n_splits = 5`) con ventana deslizante fija (`window = 26000`, `embargo = 1`), generando pronósticos puntuales en la escala original de pasajeros por franja de 15 minutos. A diferencia del ajuste estático dentro de la muestra, la actualización progresiva de la ventana deslizante incorpora el repunte del primer semestre de 2025.

In [35]:
df_eval_context = pd.concat([train, holdout], axis=0)
X_eval_all, y_eval_all, y_naive_eval_all, _ = trainer._build_dataset(df_eval_context)

holdout_mask = X_eval_all.index >= holdout.index.min()
X_holdout_final = X_eval_all.loc[holdout_mask]
y_holdout_final = y_eval_all[holdout_mask]
y_naive_holdout_final = y_naive_eval_all[holdout_mask]

start_time = time.time()
preds_holdout = model_production.forecast(
    X_seq=X_holdout_final,
    y_seq=y_holdout_final,
    window=window,
    embargo=trainer.embargo,
    n_splits=trainer.n_splits,
)
preds_time = time.time() - start_time

df_preds = pd.DataFrame({
    "date": X_holdout_final.index,
    "y_true": y_holdout_final,
    "y_naive": y_naive_holdout_final,
    "y_pred": preds_holdout,
})

output_path = RESULTS_PATH / "lgbm_holdout_predictions.csv"
df_preds.to_csv(output_path, index=False)

El análisis de los residuos fuera de muestra ($e_t = y_t - \hat{y}_t$, evaluado sobre las últimas $1000$ franjas del conjunto *holdout*) evidenció persistencia de estructura temporal y heterocedasticidad condicional en el error de pronóstico. Las pruebas de Ljung-Box rechazaron la hipótesis nula de independencia serial en todos los rezagos evaluados $\left(p_\text{adj} < .001\right)$, confirmando la presencia de autocorrelación remanente de corto y mediano plazo. De forma paralela, la prueba ARCH-LM detectó efectos ARCH altamente significativos en los tres horizontes evaluados $\left(p_\text{adj} < .001\right)$. Esto indica que, si bien el modelo *LightGBM-Tweedie* captura con precisión la envolvente diaria e intradía de la demanda esperada, persisten fluctuaciones de alta frecuencia de 15 minutos y agrupamiento de varianza residual asociados a la variabilidad operativa propia de las franjas pico, los cierres atípicos de fin de año y la relación media-varianza del proceso $\left(\operatorname{Var}(y) = \phi\,\mu^{1.4311}\right)$.

In [36]:
residuals_test = (
    df_preds["y_true"].to_numpy() - df_preds["y_pred"].to_numpy()
)

block_size = 1000
residuals_block = residuals_test[-block_size:]
std_residuals_block = (
    (residuals_block - residuals_block.mean())
    / residuals_block.std(ddof=1)
)

diag_test = tt.run_model_diagnostics(
    std_residuals_block,
    lags=(10, 20, 30),
)

tt.display_styled_df(
    df=diag_test,
    p_value_cols=["p-valor", "p-ajustado"],
    align_left_first=True,
)

Prueba,Estadístico,p-valor,p-ajustado (holm),Conclusión (α=0.05)
Ljung-Box (Lag 10),870.877,< .001,< .001,Autocorrelación Significativa
Ljung-Box (Lag 20),979.003,< .001,< .001,Autocorrelación Significativa
Ljung-Box (Lag 30),1152.747,< .001,< .001,Autocorrelación Significativa
ARCH-LM (Lag 10),288.787,< .001,< .001,Efectos ARCH Significativos
ARCH-LM (Lag 20),292.307,< .001,< .001,Efectos ARCH Significativos
ARCH-LM (Lag 30),303.947,< .001,< .001,Efectos ARCH Significativos


De acuerdo con los criterios establecidos en el protocolo experimental, no existe evidencia de sobreajuste (*overfitting*) ni de subajuste (*underfitting*) en el modelo LightGBM-Tweedie. La función de pérdida principal (Tweedie Deviance) mantiene su incremento relativo dentro del umbral de tolerancia pre-establecido, lo que apoya una adecuada capacidad de generalización fuera de muestra. Este comportamiento es consistente con la estabilidad observada en las métricas de error absoluto, cuadrático, mediano y relativo (`MAE`, `RMSE`, `MSE`, `Pinball` y `SMAPE`), donde las discrepancias entre entrenamiento y prueba resultan marginales e incluso favorables fuera de muestra.

Las mayores variaciones registradas en el `MASE` y la inversión de signo del `Bias` no responden a un deterioro estructural del modelo, sino a los cambios de régimen entre ambos periodos y a la sensibilidad direccional frente a caídas puntuales de demanda. Mientras que en entrenamiento el sesgo negativo refleja el anclaje histórico de la contracción de 2024 frente al repunte de inicios de 2025, en prueba la actualización de la ventana deslizante proyecta una envolvente laboral alta que sobrestima transitoriamente el flujo esperado en vísperas de cierre de fin de año y días puente, al tiempo que la menor dispersión del pronóstico ingenuo estacional en el segundo semestre eleva el cociente `MASE` sin comprometer la precisión del régimen operativo regular.

In [38]:
df_preds = pd.read_csv(RESULTS_PATH / "lgbm_holdout_predictions.csv")

X_train_full, y_train_full, y_naive_train_full, _ = trainer._build_dataset(train)
preds_train_final = model_production.predict_in_sample(X_train_full.iloc[-window:])

ts.display_train_test_comparison(
    y_train=y_train_full[-window:],
    preds_train=preds_train_final,
    y_naive_train=y_naive_train_full[-window:],
    y_test=df_preds["y_true"].to_numpy(),
    preds_test=df_preds["y_pred"].to_numpy(),
    y_naive_test=df_preds["y_naive"].to_numpy(),
    p=1.4311, error_relativo=True
)

Métrica,Entrenamiento (In-Sample),Prueba (Out-of-Sample),Error Relativo (%)
Tweedie Deviance,2.2458,2.2471,+0.0606%
MAE,60.1543,59.0181,-1.8888%
RMSE,83.5272,82.3706,-1.3847%
MSE,6976.7880,6784.9095,-2.7502%
Bias,-9.1904,11.4566,+224.6585%
SMAPE,0.2320,0.2256,-2.7735%
MASE,0.6043,0.6877,+13.7992%
Pinball (τ=0.50),30.0771,29.5091,-1.8888%


La inspección visual de las trayectorias predichas frente a la demanda realizada en el conjunto *holdout* muestra que el modelo LightGBM-Tweedie reproduce con alta fidelidad la envolvente diaria, la estacionalidad semanal y el perfil multimodal intradía a lo largo del segundo semestre. A escala agregada diaria (panel superior), el ajuste sigue con precisión la demanda operativa en días hábiles y corrige el sesgo a la baja observado en la ventana de entrenamiento mediante el reentrenamiento deslizante, presentando únicamente episodios puntuales de sobreestimación asociados a caídas atípicas en días festivos (como finales de junio, julio y octubre) y cierres operativos de final de diciembre.

A escala de franjas de 15 minutos (paneles inferiores, correspondientes a la primera semana de septiembre y al 15 de octubre de 2025), persiste una ligera subestimación en la cúspide de la hora pico matutina (alrededor de las 05:00–05:30, alcanzando cerca de $1000\text{--}1200$ pasajeros por franja frente a máximos reales de $1200\text{--}1300$), aunque de magnitud sustancialmente menor que la registrada durante el entrenamiento. Este comportamiento responde al efecto de suavizado propio de la estimación de la media condicional mediante árboles de decisión frente a fluctuaciones de alta frecuencia, sin comprometer la captura de la meseta de media mañana ni el repunte vespertino.

In [39]:
ts.plot_temporal_fit_panel(
    dates=pd.to_datetime(df_preds["date"]),
    y_true=df_preds["y_true"].to_numpy(),
    y_pred=df_preds["y_pred"].to_numpy(),
    zoom_left=("2025-09-01", "2025-09-07"),
    zoom_right=("2025-10-15", "2025-10-15"),
    text_left="Septiembre 2025",
    text_right="15 Octubre 2025",
    title="Demanda de pasajeros Holdout 2025 (Out-of-Sample)",
    xlab_left="Día del mes",
    xlab_right="Hora del día",
    ylab_top="Pasajeros por día",
    ylab_zoom="Pasajeros por franja",
    zoom_ylim_pad=(0.10, 0.12),
    hspace=0.1,
    wspace=0.05,
)